In [ ]:
import synapseclient

import json

import pandas as pd
import great_expectations as gx

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType
from expectations.expect_column_values_to_have_list_length_in_range import ExpectColumnValuesToHaveListLengthInRange

# Create Expectation Suite for ui_config Data

`ui_config` is a pass-through dataset shared by three applications (Agora, Model AD, xQTL). All three
name the dataset `ui_config`, so they load this one suite. Each app's `adt` run validates only its
own `ui_config.json`, so the suite is authored to be satisfiable by each file independently. We load
and concatenate all three local files here so every case is exercised during authoring.

## Get Example Data Files

In [ ]:
syn = synapseclient.Synapse()
syn.login()


## Create Validator Object on Data File

In [ ]:
# Author against the processed preprod ui_config files
# TODO update xqtl synId to processed preprod file instead of source file, once it exists
agora_df = pd.read_json(syn.get("syn73683605").path)
model_ad_df = pd.read_json(syn.get("syn73774525").path)
xqtl_df = pd.read_json(syn.get("syn77143753").path)
df = pd.concat([agora_df, model_ad_df, xqtl_df], ignore_index=True)
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, ["columns", "filters"])
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "ui_config"

In [ ]:
nested_columns = ["columns", "filters"]
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "ui_config"

## Load JSON Schemas for Nested Columns

In [ ]:
with open("../src/agoradatatools/great_expectations/gx/json_schemas/ui_config/columns_schema.json", "r") as file:
    columns_schema = json.load(file)

with open("../src/agoradatatools/great_expectations/gx/json_schemas/ui_config/filters_schema.json", "r") as file:
    filters_schema = json.load(file)

## Add Expectations to Validator Object

The suite is uniform across all three apps (no page-conditioning): `columns` and `filters` are
validated by single shared JSON schemas; `page`, `dropdowns`, and `row_count` by native checks.

In [ ]:
# page (union of all three apps' page/interface identifiers)
validator.expect_column_values_to_be_of_type("page", "str")
validator.expect_column_values_to_not_be_null("page")
validator.expect_column_values_to_be_in_set(
    "page",
    [
        "Nominated Targets", "Nominated Drugs",
        "Differential Expression", "Disease Correlation", "Model Overview", "Marmoset Model Overview",
        "eQTLs",
    ],
)

In [ ]:
# columns / filters (nested): validated structurally by the shared JSON schemas
validator.expect_column_values_to_match_json_schema("columns", json_schema=columns_schema)
validator.expect_column_values_to_match_json_schema("filters", json_schema=filters_schema)

In [ ]:
# dropdowns (list of scalars, un-nested): empty in Agora/xQTL, populated with strings in Model AD
validator.expect_column_values_to_be_of_type("dropdowns", "list")
validator.expect_column_values_to_not_be_null("dropdowns")
validator.expect_column_values_to_have_list_members_of_type(column="dropdowns", member_type="str")
validator.expect_column_values_to_have_list_length_in_range(column="dropdowns", list_length_range=[0, 2])

In [ ]:
# row_count is present in all three apps and always null today; tighten to that contract so a future
# populated value fails loudly and prompts a suite update.
validator.expect_column_values_to_be_null("row_count")

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()